# Mini RAG — Demo en Vivo
**RAG = Retrieval Augmented Generation**

El problema: los LLMs tienen una **fecha de corte** — no saben nada de eventos posteriores ni de información privada.

La solución RAG en 3 pasos:
1. **Retrieve** — Buscar el documento más relevante para la pregunta
2. **Augment** — Construir un prompt con ese documento como contexto
3. **Generate** — El LLM responde usando ese contexto

In [11]:
!pip install openai -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np
from openai import OpenAI

OPENAI_API_KEY = "sk-..."  # Reemplaza con tu clave
client = OpenAI(api_key=OPENAI_API_KEY)

## Paso 0 — El problema: knowledge cutoff

Pregunta al LLM algo que no puede saber. Así entendemos **por qué** necesitamos RAG.

In [12]:
preguntas_imposibles = [
    "¿Quién ganó el Mundial 2026?",
    "¿Cuál es el precio de la acción de Apple hoy?",
    "¿Qué dice el informe interno Q1 2026 de mi empresa?",
]

print("Lo que responde el LLM sin contexto externo:\n")
for p in preguntas_imposibles:
    r = client.responses.create(model="gpt-4o-mini", input=p)
    print(f"  Q: {p}")
    print(f"  A: {r.output_text.strip()[:150]}")
    print()

Lo que responde el LLM sin contexto externo:

  Q: ¿Quién ganó el Mundial 2026?
  A: No puedo proporcionar información sobre eventos futuros o resultados que ocurran después de mi última actualización en 2023. El Mundial de 2026 está p

  Q: ¿Cuál es el precio de la acción de Apple hoy?
  A: Lo siento, pero no puedo proporcionar información en tiempo real, incluidos los precios de las acciones. Te recomiendo que consultes un sitio web de f

  Q: ¿Qué dice el informe interno Q1 2026 de mi empresa?
  A: Lo siento, pero no tengo acceso a informes internos ni a información específica de tu empresa. Si necesitas ayuda para analizar el informe o con un as



## Paso 1 — Nuestra "base de conocimiento"
Información que el LLM **no puede saber** (el Mundial 2026 aún no ha ocurrido en su entrenamiento)

In [13]:
documentos = [
    # Resultados generales
    "España ganó el Mundial 2026 disputado en Estados Unidos, México y Canadá. Fue su cuarta estrella.",
    "La final del Mundial 2026 se jugó en el MetLife Stadium de Nueva York. España venció a Brasil 2-1 en la prórroga.",
    "El tercer puesto del Mundial 2026 fue para Francia, que derrotó a Argentina 3-2 en el partido por el bronce.",
    "El Mundial 2026 fue el primero en contar con 48 selecciones. Se disputaron 104 partidos en total.",

    # Jugadores individuales
    "Lamine Yamal fue elegido Balón de Oro del Mundial 2026 con 18 años, convirtiéndose en el más joven de la historia.",
    "El máximo goleador del Mundial 2026 fue Vinicius Jr. con 8 goles, aunque Brasil cayó en la final ante España.",
    "Pedri marcó el gol de la victoria en la prórroga de la final del Mundial 2026 con un disparo de larga distancia.",
    "El portero David Raya fue elegido mejor guardameta del torneo tras encajar solo 2 goles en 7 partidos.",

    # Selecciones
    "La selección española llegó a la final del Mundial 2026 sin perder ningún partido, con 6 victorias y 1 empate.",
    "Alemania fue eliminada en cuartos de final por Brasil en los penaltis. Fue su peor resultado desde 2018.",
    "Marruecos llegó por segunda vez a semifinales de un Mundial, cayendo ante España por 1-0.",
    "La selección de Estados Unidos, como anfitriona, llegó a octavos de final antes de ser eliminada por México.",
]

print(f"{len(documentos)} documentos cargados")

12 documentos cargados


In [14]:
# Indexar: convertir cada documento a un vector de embedding (se hace UNA vez)
def get_embedding(text):
    r = client.embeddings.create(model="text-embedding-3-small", input=text)
    return r.data[0].embedding

print("Indexando documentos...")
doc_embeddings = [get_embedding(doc) for doc in documentos]
print(f"{len(doc_embeddings)} documentos indexados ✓")

Indexando documentos...
12 documentos indexados ✓


## Paso 2 — Retrieve: buscar por similitud semántica

Cada documento y la pregunta se convierten en un **vector de embeddings**.  
Recuperamos los `top_k` documentos cuyo vector sea más cercano al de la pregunta (**similitud coseno**).

In [15]:
def recuperar(pregunta, top_k=2):
    q_emb = np.array(get_embedding(pregunta))
    # Los embeddings de OpenAI están normalizados → dot product = similitud coseno
    scores = [np.dot(q_emb, np.array(e)) for e in doc_embeddings]

    ranking = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

    print(f"Pregunta: '{pregunta}'\n")
    print("Ranking por similitud semántica:")
    for pos, (idx, score) in enumerate(ranking):
        marca = " ✓" if pos < top_k else ""
        print(f"  sim={score:.3f}{marca}  {documentos[idx][:70]}...")

    top_docs = [documentos[idx] for idx, _ in ranking[:top_k]]
    print(f"\n  → Top {top_k} seleccionados\n")
    return "\n".join(top_docs)

# Prueba
recuperar("¿Quién fue el máximo goleador del Mundial?")

Pregunta: '¿Quién fue el máximo goleador del Mundial?'

Ranking por similitud semántica:
  sim=0.606 ✓  El máximo goleador del Mundial 2026 fue Vinicius Jr. con 8 goles, aunq...
  sim=0.426 ✓  Lamine Yamal fue elegido Balón de Oro del Mundial 2026 con 18 años, co...
  sim=0.405  Pedri marcó el gol de la victoria en la prórroga de la final del Mundi...
  sim=0.398  El Mundial 2026 fue el primero en contar con 48 selecciones. Se disput...
  sim=0.389  Marruecos llegó por segunda vez a semifinales de un Mundial, cayendo a...
  sim=0.373  La selección española llegó a la final del Mundial 2026 sin perder nin...
  sim=0.362  El portero David Raya fue elegido mejor guardameta del torneo tras enc...
  sim=0.360  España ganó el Mundial 2026 disputado en Estados Unidos, México y Cana...
  sim=0.348  Alemania fue eliminada en cuartos de final por Brasil en los penaltis....
  sim=0.348  El tercer puesto del Mundial 2026 fue para Francia, que derrotó a Arge...
  sim=0.325  La final del Mundial 202

'El máximo goleador del Mundial 2026 fue Vinicius Jr. con 8 goles, aunque Brasil cayó en la final ante España.\nLamine Yamal fue elegido Balón de Oro del Mundial 2026 con 18 años, convirtiéndose en el más joven de la historia.'

## Paso 3 — Augment + Generate: construir el prompt e invocar el LLM

El truco de RAG: meter los documentos recuperados en el **system prompt**.  
El LLM nunca ve los documentos directamente — solo recibe texto.

In [16]:
def preguntar_con_rag(pregunta, top_k=2, temperatura=0.0, verbose=True):
    contexto = recuperar(pregunta, top_k=top_k)

    system_prompt = (
        "Eres un asistente experto en el Mundial 2026. "
        "Responde SOLO usando la información proporcionada. "
        "Si no está en el contexto, di que no lo sabes.\n\n"
        f"CONTEXTO:\n{contexto}"
    )

    if verbose:
        print("=" * 60)
        print("PROMPT COMPLETO QUE RECIBE EL LLM:")
        print(f"\n[SYSTEM]:\n{system_prompt}")
        print(f"\n[USER]: {pregunta}")
        print("=" * 60 + "\n")

    r = client.responses.create(
        model="gpt-4o-mini",
        instructions=system_prompt,
        input=pregunta,
        temperature=temperatura
    )
    return r.output_text

# Prueba con prompt visible
respuesta = preguntar_con_rag("¿Dónde se jugó la final?", verbose=True)
print("RESPUESTA:", respuesta)

Pregunta: '¿Dónde se jugó la final?'

Ranking por similitud semántica:
  sim=0.533 ✓  La final del Mundial 2026 se jugó en el MetLife Stadium de Nueva York....
  sim=0.402 ✓  El tercer puesto del Mundial 2026 fue para Francia, que derrotó a Arge...
  sim=0.372  España ganó el Mundial 2026 disputado en Estados Unidos, México y Cana...
  sim=0.372  Pedri marcó el gol de la victoria en la prórroga de la final del Mundi...
  sim=0.361  La selección española llegó a la final del Mundial 2026 sin perder nin...
  sim=0.359  Alemania fue eliminada en cuartos de final por Brasil en los penaltis....
  sim=0.355  Marruecos llegó por segunda vez a semifinales de un Mundial, cayendo a...
  sim=0.345  La selección de Estados Unidos, como anfitriona, llegó a octavos de fi...
  sim=0.327  El máximo goleador del Mundial 2026 fue Vinicius Jr. con 8 goles, aunq...
  sim=0.304  El portero David Raya fue elegido mejor guardameta del torneo tras enc...
  sim=0.263  El Mundial 2026 fue el primero en contar c

## Demo — SIN RAG vs CON RAG (varias preguntas)

In [17]:
preguntas = [
    "¿Quién ganó el Mundial 2026?",
    "¿Quién fue el Balón de Oro?",
    "¿Qué pasó con Alemania?",
    "¿Cuántos goles marcó Vinicius?",
]

for pregunta in preguntas:
    print(f"\n{'─'*55}")
    print(f"PREGUNTA: {pregunta}")

    sin = client.responses.create(model="gpt-4o-mini", input=pregunta)
    print(f"  SIN RAG: {sin.output_text.strip()[:120]}")

    con = preguntar_con_rag(pregunta, verbose=False)
    print(f"  CON RAG: {con.strip()}")


───────────────────────────────────────────────────────
PREGUNTA: ¿Quién ganó el Mundial 2026?
  SIN RAG: No puedo proporcionar información actualizada sobre el ganador del Mundial 2026, ya que mi conocimiento se detiene en 20
Pregunta: '¿Quién ganó el Mundial 2026?'

Ranking por similitud semántica:
  sim=0.666 ✓  España ganó el Mundial 2026 disputado en Estados Unidos, México y Cana...
  sim=0.581 ✓  El Mundial 2026 fue el primero en contar con 48 selecciones. Se disput...
  sim=0.572  La final del Mundial 2026 se jugó en el MetLife Stadium de Nueva York....
  sim=0.528  La selección española llegó a la final del Mundial 2026 sin perder nin...
  sim=0.513  El tercer puesto del Mundial 2026 fue para Francia, que derrotó a Arge...
  sim=0.478  Pedri marcó el gol de la victoria en la prórroga de la final del Mundi...
  sim=0.465  Lamine Yamal fue elegido Balón de Oro del Mundial 2026 con 18 años, co...
  sim=0.456  El máximo goleador del Mundial 2026 fue Vinicius Jr. con 8 goles, aunq.

## Extra 1 — Temperature: creatividad vs determinismo

`temperature=0.0` → respuesta determinista y factual (ideal para RAG)  
`temperature=1.5` → respuesta más creativa y variable (útil para generación de texto)

In [9]:
pregunta = "¿Cómo describirías la victoria de España en el Mundial 2026?"

print("── Temperature = 0.0 (determinista) ──")
print(preguntar_con_rag(pregunta, temperatura=0.0, verbose=False))

print("\n── Temperature = 1.5 (creativo) ──")
print(preguntar_con_rag(pregunta, temperatura=1.5, verbose=False))

print("\n── Temperature = 1.5 (segunda vez — resultado diferente) ──")
print(preguntar_con_rag(pregunta, temperatura=1.5, verbose=False))

── Temperature = 0.0 (determinista) ──
Pregunta: '¿Cómo describirías la victoria de España en el Mundial 2026?'

Ranking por similitud semántica:
  sim=0.620 ✓  La selección española llegó a la final del Mundial 2026 sin perder nin...
  sim=0.603 ✓  España ganó el Mundial 2026 disputado en Estados Unidos, México y Cana...
  sim=0.544  La final del Mundial 2026 se jugó en el MetLife Stadium de Nueva York....
  sim=0.485  Pedri marcó el gol de la victoria en la prórroga de la final del Mundi...
  sim=0.460  El máximo goleador del Mundial 2026 fue Vinicius Jr. con 8 goles, aunq...
  sim=0.444  El tercer puesto del Mundial 2026 fue para Francia, que derrotó a Arge...
  sim=0.428  El Mundial 2026 fue el primero en contar con 48 selecciones. Se disput...
  sim=0.411  Marruecos llegó por segunda vez a semifinales de un Mundial, cayendo a...
  sim=0.400  Lamine Yamal fue elegido Balón de Oro del Mundial 2026 con 18 años, co...
  sim=0.324  Alemania fue eliminada en cuartos de final por Brasil 

## Extra 2 — Limitaciones: cuándo falla el keyword matching

El conteo de palabras no entiende **significado**. Si la pregunta usa sinónimos o lenguaje indirecto, puede recuperar el documento equivocado.

Solución real: **embeddings** (vectores semánticos) → el Lab de RAG avanzado.

In [18]:
preguntas_dificiles = [
    "¿Quién metió el tanto definitivo en la final?",   # sinónimo de "gol" → keyword matching fallaría
    "¿Qué selección europea quedó tercera?",           # razonamiento: Francia es europea, no lo dice explícito
    "¿Quién paró más en el torneo?",                   # "paró" ≈ "guardameta" → semántica pura
]

print("Preguntas que engañan al keyword matching pero NO a los embeddings:\n")
for p in preguntas_dificiles:
    print(f"Pregunta: {p}")
    recuperar(p, top_k=1)
    print()

Preguntas que engañan al keyword matching pero NO a los embeddings:

Pregunta: ¿Quién metió el tanto definitivo en la final?
Pregunta: '¿Quién metió el tanto definitivo en la final?'

Ranking por similitud semántica:
  sim=0.445 ✓  Pedri marcó el gol de la victoria en la prórroga de la final del Mundi...
  sim=0.420  La final del Mundial 2026 se jugó en el MetLife Stadium de Nueva York....
  sim=0.409  Marruecos llegó por segunda vez a semifinales de un Mundial, cayendo a...
  sim=0.404  El máximo goleador del Mundial 2026 fue Vinicius Jr. con 8 goles, aunq...
  sim=0.381  El tercer puesto del Mundial 2026 fue para Francia, que derrotó a Arge...
  sim=0.373  La selección española llegó a la final del Mundial 2026 sin perder nin...
  sim=0.366  Alemania fue eliminada en cuartos de final por Brasil en los penaltis....
  sim=0.363  El portero David Raya fue elegido mejor guardameta del torneo tras enc...
  sim=0.360  La selección de Estados Unidos, como anfitriona, llegó a octavos de fi..